# 00 · Framing the problem

> *"Most failed recommender projects fail before the first line of model code. They optimise the wrong thing, on the wrong data, measured the wrong way."*

**Learning objectives**

By the end of this notebook you should be able to:

1. State the business problem a recommender solves, and what "good" means for it.
2. Explain the difference between **explicit** and **implicit** feedback, and why implicit is harder.
3. Describe the **offline → online** metric hierarchy and why offline metrics are only a proxy.
4. Find your way around the rest of this project.

## 1. The scenario

You have just joined the data science team at an e-commerce retailer. The site logs every **view**, **add-to-cart** and **purchase**. Product wants a *"Recommended for you"* carousel on the home page: **10 products, personalised, refreshed daily.**

We will use the public [RetailRocket e-commerce dataset](https://www.kaggle.com/datasets/retailrocket/ecommerce-dataset) from Kaggle. It holds 4.5 months of real, anonymised behaviour from a real shop, and it has every awkward property a real dataset has: extreme sparsity, bots, a long tail, and lots of one-visit users.

### Questions a Principal DS asks *before* modelling

| Question | Why it matters | Our answer (for this project) |
|---|---|---|
| What decision does the model drive? | Defines the output format | Rank 10 items per user, once a day |
| Who are the users? | Cold-start strategy | Returning visitors get personalised recs; new ones get trending items |
| What is "success"? | Choice of target and metric | More add-to-carts and purchases from the carousel |
| What data exists at serving time? | Prevents leakage | Only events up to "now" |
| What are the constraints? | Model choice | Batch scoring, < 1h nightly, explainable to merchandisers |
| What is the baseline today? | Shows whether the model is worth building | Probably "best sellers": we must beat it |

## 2. Explicit vs implicit feedback

**Explicit feedback** is a user *telling* you what they think: star ratings or thumbs up/down (MovieLens, Netflix Prize).

**Implicit feedback** is what we have here: we *infer* interest from behaviour. It is far more abundant (every visitor generates it) but much harder to interpret:

| | Explicit | Implicit |
|---|---|---|
| Volume | Low: few users rate | High: every click is logged |
| Negative signal | Yes (1 star) | **No.** A missing interaction could be dislike *or* "never saw it" |
| Noise | Low | High: accidental clicks, bots, gift purchases |
| Signal strength | Graded | Varies by event type: view < add-to-cart < purchase |

The missing-negatives problem shapes everything that follows:

* We cannot treat "no interaction" as "disliked". Models such as **implicit ALS** (notebook 05) instead treat it as *low-confidence* negative evidence.
* **BPR** sidesteps it by learning only *relative* preferences: "this user prefers an item they interacted with over a random one".
* Evaluation is biased: an item missing from the test set may have been relevant but simply never shown.

## 3. How we will measure success

There is a hierarchy, and only the top level is what the business actually wants:

```
  Business KPI      revenue, conversion rate, retention          ← what we want
       ▲
  Online metrics    carousel CTR, add-to-cart rate (A/B test)    ← what we can test
       ▲
  Offline metrics   recall@10, NDCG@10 on held-out future data   ← what we iterate on daily
```

Offline metrics are cheap and fast, so we use them to choose between models. But they are a **proxy**: a model that wins offline is a *candidate* for an A/B test, not a guaranteed winner. Notebook 07 discusses why the two can disagree.

**Our offline protocol** (built in notebooks 02–03):

* Train on everything before a cutoff date, and test on the following 7 days. This mirrors how the model is used in production.
* For each returning user, recommend 10 items they have **not** interacted with before.
* Score the list against what they actually interacted with in the test week.
* Primary metric: **NDCG@10**, which rewards relevant items near the top. Secondary metrics: recall@10, hit-rate@10, catalogue coverage and novelty.

## 4. The roadmap

| Notebook | Topic | Key idea |
|---|---|---|
| **00** | Framing (you are here) | Define the problem before the model |
| **01** | Exploratory data analysis | Sparsity, long tail, funnel, bots |
| **02** | Preprocessing & splitting | Event weights, k-core, *temporal* splits, leakage |
| **03** | Baselines & metrics | Metrics by hand; popularity is a strong baseline |
| **04** | Item-kNN | "People who viewed this also viewed", explainability |
| **05** | Matrix factorisation | ALS for implicit data, BPR, embeddings |
| **06** | Two-stage ranking | Candidate generation + learning-to-rank |
| **07** | Evaluation & production | Segments, significance, bias, A/B tests, serving |

All the logic lives in the `recsys` package (`src/recsys/`). The notebooks call it so that every piece of code is tested once (`tests/`) and reused everywhere. **Read the source.** It is written to be read, and each module's docstring explains the *why*.

## 5. First look at the data

If you have run `make data`, this loads the real RetailRocket events. Otherwise it falls back to a **synthetic** log with the same schema and prints a loud warning. Everything still runs, but the numbers are not real.

In [1]:
# Make `import recsys` work even without `pip install -e .`
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().resolve().parent / "src"))

from recsys.data import load_or_synthesize

events, item_categories, SOURCE = load_or_synthesize()
events.head(10)

  USING SYNTHETIC DATA: data/raw/events.csv was not found.
  The numbers below come from a simulator, NOT from RetailRocket.
  Run `make data` (needs a Kaggle API token) to use the real dataset.


,timestamp,visitorid,event,itemid,transactionid
0,2015-05-03 00:01:30.872,18285,view,469,NaN
1,2015-05-03 00:03:40.791,28666,view,4274,NaN
2,2015-05-03 00:04:57.406,16230,view,12165,NaN
3,2015-05-03 00:09:44.663,2608,view,5592,NaN
4,2015-05-03 00:10:48.095,2608,view,11413,NaN
5,2015-05-03 00:11:35.826,16799,view,12795,NaN
6,2015-05-03 00:11:41.062,2608,view,8885,NaN
7,2015-05-03 00:13:02.188,16230,view,1509,NaN
8,2015-05-03 00:13:05.327,13710,view,1052,NaN
9,2015-05-03 00:13:07.339,2608,view,11668,NaN


In [2]:
events.info(memory_usage="deep")

<class 'pandas.DataFrame'>
RangeIndex: 71281 entries, 0 to 71280
Data columns (total 5 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   timestamp      71281 non-null  datetime64[ns]
 1   visitorid      71281 non-null  int64         
 2   event          71281 non-null  category      
 3   itemid         71281 non-null  int64         
 4   transactionid  882 non-null    float64       
dtypes: category(1), datetime64[ns](1), float64(1), int64(2)
memory usage: 2.2 MB


In [3]:
print(f"Date range : {events['timestamp'].min()}  →  {events['timestamp'].max()}")
print(f"Visitors   : {events['visitorid'].nunique():,}")
print(f"Items      : {events['itemid'].nunique():,}")
print(events["event"].value_counts().to_string())

Date range : 2015-05-03 00:01:30.872000  →  2015-07-01 23:59:48.921000
Visitors   : 3,000
Items      : 1,500
event
view           68116
addtocart       2283
transaction      882


## Key takeaways

* Start from the **decision** and the **success metric**, not the algorithm.
* Implicit feedback has **no true negatives**. Every modelling and evaluation choice has to account for that.
* Offline metrics choose candidates, and online experiments choose winners.

## Exercises

1. **(Easy)** Suppose the carousel instead appeared on the *cart page*. How would the success metric and the definition of "relevant" change?
2. **(Medium)** List three ways a recommender could raise CTR but *hurt* revenue. Which guardrail metrics would catch each one?
3. **(Stretch)** Read Hu, Koren & Volinsky (2008), *Collaborative Filtering for Implicit Feedback Datasets*, sections 1–3. Summarise their preference/confidence split in two sentences.